In [1]:
!pip install torch

In [2]:
import torch

gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {vram:.1f} GB")

assert "A100" in gpu_name or vram > 30, "ERROR: Expected A100 (40GB+)"

from google.colab import drive
drive.mount("/content/drive")

import os
BASE_DIR = "/content/drive/MyDrive/easydistill_run"
os.makedirs(BASE_DIR, exist_ok=True)
print(f"Base dir: {BASE_DIR}")

GPU : NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Mounted at /content/drive
Base dir: /content/drive/MyDrive/easydistill_run


In [3]:
!git clone https://github.com/modelscope/easydistill /content/easydistill
%cd /content/easydistill
!pip install -e . -q
!pip install -r /content/easydistill/requirements.txt -q
!pip install vllm -q

Cloning into '/content/easydistill'...
remote: Enumerating objects: 1191, done.
remote: Counting objects: 100% (318/318), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 1191 (delta 233), reused 212 (delta 172), pack-reused 873 (from 2)
Receiving objects: 100% (1191/1191), 54.22 MiB | 19.18 MiB/s, done.
Resolving deltas: 100% (452/452), done.
Updating files: 100% (512/512), done.
/content/easydistill
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 121.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 151.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.0/348.0 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.4/326.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.4/98.4 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━

In [15]:
import os

BASE_DIR     = "/content/drive/MyDrive/easydistill_run"
DATA_DIR     = f"{BASE_DIR}/data"
LOGITS_DIR   = f"{BASE_DIR}/teacher_logits"
STAGE1_DIR   = f"{BASE_DIR}/result_stage1"   # black-box SFT output
STAGE2_DIR   = f"{BASE_DIR}/result_stage2"   # white-box KD output  ← final model
TEMPLATE_DIR = f"{BASE_DIR}/templates"

for d in [DATA_DIR, LOGITS_DIR, STAGE1_DIR, STAGE2_DIR, TEMPLATE_DIR]:
    os.makedirs(d, exist_ok=True)

dataset_path  = f"{DATA_DIR}/code_generation_dataset.json"
logits_path   = f"{LOGITS_DIR}/logits.json"
template_path = f"{TEMPLATE_DIR}/chat_template_kd.jinja"
stage1_config = f"{BASE_DIR}/stage1.json"
stage2_config = f"{BASE_DIR}/stage2.json"

print("Paths:")
for k, v in {
    "Dataset" : dataset_path,
    "Logits"  : logits_path,
    "Template": template_path,
    "Stage1"  : STAGE1_DIR,
    "Stage2"  : STAGE2_DIR,
}.items():
    print(f"  {k:10s}: {v}")

✅ Paths:
  Dataset   : /content/drive/MyDrive/easydistill_run/data/code_generation_dataset.json
  Logits    : /content/drive/MyDrive/easydistill_run/teacher_logits/logits.json
  Template  : /content/drive/MyDrive/easydistill_run/templates/chat_template_kd.jinja
  Stage1    : /content/drive/MyDrive/easydistill_run/result_stage1
  Stage2    : /content/drive/MyDrive/easydistill_run/result_stage2


In [6]:
import json
from datasets import load_dataset

ds = load_dataset("sahil2801/CodeAlpaca-20k", split="train")

records = []
for sample in ds:
    instruction = sample["instruction"]
    if sample["input"].strip():
        instruction += "\n\n" + sample["input"]
    records.append({
        "instruction": instruction,
        "output": sample["output"]
    })

with open(dataset_path, "w") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

# Sanity check
with open(dataset_path) as f:
    test = json.load(f)

print(f"{len(test)} records saved → {dataset_path}")
print(f"   Keys   : {list(test[0].keys())}")
print(f"   Sample : {test[0]['instruction'][:80]}...")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

✅ 20022 records saved → /content/drive/MyDrive/easydistill_run/data/code_generation_dataset.json
   Keys   : ['instruction', 'output']
   Sample : Create an array of length 5 which contains all even numbers between 1 and 10....


In [7]:
# Matches the format the recipe README expects
with open(template_path, "w") as f:
    f.write(
        "<|im_start|>system\n"
        "You are a coding assistant.<|im_end|>\n"
        "<|im_start|>user\n"
        "{{ instruction }}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

print(f"Template saved → {template_path}")
with open(template_path) as f:
    print(f.read())

✅ Template saved → /content/drive/MyDrive/easydistill_run/templates/chat_template_kd.jinja
<|im_start|>system
You are a coding assistant.<|im_end|>
<|im_start|>user
{{ instruction }}<|im_end|>
<|im_start|>assistant



In [8]:
import json
from huggingface_hub import snapshot_download

# Download config only — just for vocab_size lookup in train.py
teacher_config_path = snapshot_download(
    "Qwen/Qwen2.5-7B-Instruct",
    ignore_patterns=["*.safetensors", "*.bin"]
)

# vLLM gets the HuggingFace name — it will download weights itself
TEACHER_HF_NAME    = "Qwen/Qwen2.5-7B-Instruct"
TEACHER_CONFIG_DIR = teacher_config_path   # config only, for vocab_size

with open(f"{teacher_config_path}/config.json") as f:
    vocab_size = json.load(f)["vocab_size"]

print(f"Teacher config dir : {teacher_config_path}")
print(f"Teacher HF name    : {TEACHER_HF_NAME}")
print(f"vocab_size         : {vocab_size}")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

✅ Teacher config dir : /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28
✅ Teacher HF name    : Qwen/Qwen2.5-7B-Instruct
✅ vocab_size         : 152064


In [9]:
import json

stage1 = {
    "job_type": "kd_black_box_api",
    "dataset": {
        "labeled_path": dataset_path,
        "template": template_path,
        "seed": 42
    },
    "models": {
        "student": "Qwen/Qwen2.5-0.5B-Instruct"
    },
    "training": {
        "output_dir": STAGE1_DIR,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "save_steps": 1000,
        "logging_steps": 1,
        "learning_rate": 2e-5,
        "weight_decay": 0.05,
        "warmup_ratio": 0.1,
        "lr_scheduler_type": "cosine",
        "bf16": True
    }
}

with open(stage1_config, "w") as f:
    json.dump(stage1, f, indent=2)

print(f"Stage 1 config → {stage1_config}")
print(json.dumps(stage1, indent=2))

✅ Stage 1 config → /content/drive/MyDrive/easydistill_run/stage1.json
{
  "job_type": "kd_black_box_api",
  "dataset": {
    "labeled_path": "/content/drive/MyDrive/easydistill_run/data/code_generation_dataset.json",
    "template": "/content/drive/MyDrive/easydistill_run/templates/chat_template_kd.jinja",
    "seed": 42
  },
  "models": {
    "student": "Qwen/Qwen2.5-0.5B-Instruct"
  },
  "training": {
    "output_dir": "/content/drive/MyDrive/easydistill_run/result_stage1",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "save_steps": 1000,
    "logging_steps": 1,
    "learning_rate": 2e-05,
    "weight_decay": 0.05,
    "warmup_ratio": 0.1,
    "lr_scheduler_type": "cosine",
    "bf16": true
  }
}


In [16]:
stage2 = {
    "job_type": "kd_white_box",
    "dataset": {
        "labeled_path": dataset_path,
        "instruction_path": dataset_path,
        "logits_path":  f"{LOGITS_DIR}/logits.jsonl",       # directory — used by DistillSFTTrainer
        "template":     template_path,
        "seed": 42
    },
    "inference": {
        "enable_chunked_prefill": True,
        "seed": 777,
        "gpu_memory_utilization": 0.45,   # leave room for student training
        "temperature": 0.8,
        "trust_remote_code": True,
        "enforce_eager": False,
        "max_model_len": 512,
        "max_new_tokens": 512,
        "top_logits_num": 10
    },
    "distillation": {
        "kd_ratio": 0.1,                  # matches official recipe
        "max_seq_length": 512,
        "distillation_type": "forward_kld"
    },
    "models": {
        "teacher": TEACHER_CONFIG_DIR,    # local path so train.py can read config.json
        "student": "Qwen/Qwen2.5-0.5B-Instruct"             # output of Stage 1 is the student for Stage 2
    },
    "training": {
        "output_dir": STAGE2_DIR,         # ← final model saved here on Drive
        "num_train_epochs": 3,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "save_steps": 1000,
        "logging_steps": 1,
        "learning_rate": 2e-5,
        "weight_decay": 0.05,
        "warmup_ratio": 0.1,
        "lr_scheduler_type": "cosine",
        "bf16": True
    }
}

with open(stage2_config, "w") as f:
    json.dump(stage2, f, indent=2)

print(f"Stage 2 config → {stage2_config}")
print(json.dumps(stage2, indent=2))

✅ Stage 2 config → /content/drive/MyDrive/easydistill_run/stage2.json
{
  "job_type": "kd_white_box",
  "dataset": {
    "labeled_path": "/content/drive/MyDrive/easydistill_run/data/code_generation_dataset.json",
    "instruction_path": "/content/drive/MyDrive/easydistill_run/data/code_generation_dataset.json",
    "logits_path": "/content/drive/MyDrive/easydistill_run/teacher_logits/logits.jsonl",
    "template": "/content/drive/MyDrive/easydistill_run/templates/chat_template_kd.jinja",
    "seed": 42
  },
  "inference": {
    "enable_chunked_prefill": true,
    "seed": 777,
    "gpu_memory_utilization": 0.45,
    "temperature": 0.8,
    "trust_remote_code": true,
    "enforce_eager": false,
    "max_model_len": 512,
    "max_new_tokens": 512,
    "top_logits_num": 10
  },
  "distillation": {
    "kd_ratio": 0.1,
    "max_seq_length": 512,
    "distillation_type": "forward_kld"
  },
  "models": {
    "teacher": "/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/

In [11]:
!pip install jsonlines -q

In [ ]:
print("Stage 1 — Black-box SFT training (~1-2 hrs on A100)...")
!python /content/easydistill/easydistill/kd/train.py --config {stage1_config}

🚀 Stage 1 — Black-box SFT training (~1-2 hrs on A100)...
2026-04-27 04:52:18.058207: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-27 04:52:18.075522: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777265538.096657    6780 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777265538.103147    6780 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777265538.119373    6780 computation_placer.cc:177] computation placer al

In [ ]:
import os

logits_file = f"{LOGITS_DIR}/logits.jsonl"

if os.path.exists(logits_file):
    with open(logits_file) as f:
        lines = sum(1 for _ in f)
    size_mb = os.path.getsize(logits_file) / 1e6
    print(f"Logits already exist — {lines:,} records, {size_mb:.0f} MB. Skipping.")
else:
    print("Stage 2a — Generating teacher logits (~90 mins on A100)...")
    !python /content/easydistill/easydistill/kd/infer.py --config {stage2_config}

🚀 Stage 2a — Generating teacher logits (~90 mins on A100)...
INFO 04-27 06:27:46 [__init__.py:239] Automatically detected platform cuda.
2026-04-27 06:27:47.459131: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-27 06:27:47.476704: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777271267.497842   32664 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777271267.504330   32664 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 

In [17]:
print("Stage 2b — White-box KD training (~1-2 hrs on A100)...")
!python /content/easydistill/easydistill/kd/train.py --config {stage2_config}

🚀 Stage 2b — White-box KD training (~1-2 hrs on A100)...
2026-04-27 12:30:09.558094: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-27 12:30:09.576346: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777293009.597885    5426 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777293009.604421    5426 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777293009.621194    5426 computation_placer.cc:177] computation placer al

In [18]:
import os

print(f"Final model location: {STAGE2_DIR}\n")

for fname in sorted(os.listdir(STAGE2_DIR)):
    size_mb = os.path.getsize(os.path.join(STAGE2_DIR, fname)) / 1e6
    print(f"  {fname:45s} {size_mb:.1f} MB")

print(f"\n Distilled Qwen2.5-0.5B saved to Google Drive at:\n   {STAGE2_DIR}")

Final model location: /content/drive/MyDrive/easydistill_run/result_stage2

  added_tokens.json                             0.0 MB
  checkpoint-1000                               0.0 MB
  checkpoint-2000                               0.0 MB
  checkpoint-3000                               0.0 MB
  checkpoint-3753                               0.0 MB
  config.json                                   0.0 MB
  generation_config.json                        0.0 MB
  merges.txt                                    1.7 MB
  model.safetensors                             1976.2 MB
  runs                                          0.0 MB
  special_tokens_map.json                       0.0 MB
  tokenizer.json                                11.4 MB
  tokenizer_config.json                         0.0 MB
  training_args.bin                             0.0 MB
  vocab.json                                    2.8 MB

✅ Distilled Qwen2.5-0.5B saved to Google Drive at:
   /content/drive/MyDrive/easydistill_run/r